# Solving CartPole-v1 with PPO (from scratch)

This notebook demonstrates how to solve the CartPole-v1 environment using a custom implementation of the Proximal Policy Optimization (PPO) algorithm in PyTorch. No external RL libraries are used.

---

**Note:** All environment and hyperparameter settings are now collected in a single `CONFIG` dictionary at the top of the notebook. To change the environment or any hyperparameter, simply edit the values in the config cell.

## 1. Install and Import Required Libraries
We will use gymnasium and torch for this implementation.

In [1]:
%pip install gymnasium[classic-control] stable-baselines3 wandb tsilva-notebook-utils==0.0.104 --quiet

zsh:1: no matches found: gymnasium[classic-control]
Note: you may need to restart the kernel to use updated packages.


In [2]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY'
])

In [3]:
from wandb import login
login()

wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import torch

def get_default_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device("mps")
    elif torch.cuda.is_available(): return torch.device("cuda")
    else: return torch.device("cpu")

DEVICE = get_default_device()
print(f"Device set to: {DEVICE}")

Device set to: mps


## 2. Set Up CartPole-v1 Environment
We will initialize the CartPole-v1 environment and display its basic information.

In [5]:
import torch.nn as nn
from tsilva_notebook_utils.gymnasium import build_env as _build_env, set_random_seed

# --- Config dictionary for all hyperparameters and environment settings ---
def setup_config(env_id):
    common = dict(
        env_id=env_id,         # Environment name
        seed=42,               # Random seed for reproducibility
        gamma=0.99,            # Discount factor for future rewards
        lam=0.95,              # GAE lambda for advantage estimation
        clip_epsilon=0.2,      # PPO clip range for policy update
        minibatch_size=64,     # Minibatch size for SGD
        episodes_per_epoch=20, # Number of episodes per training epoch
        eval_interval=5,       # Evaluate every N epochs
        eval_episodes=20,      # Number of episodes for evaluation
        reward_threshold=200,  # Reward threshold to consider environment solved
        policy_lr=3e-4,        # Learning rate for policy network
        value_lr=1e-3,         # Learning rate for value network
        hidden_dim=64,         # Hidden layer size for networks
        entropy_coef=0.01,     # Coefficient for entropy bonus (encourages exploration)
        normalize=False,       # Whether to use input normalization
        updates_per_epoch=10,      # Number of epochs to update policy per training step
        n_envs="auto"          # Maximum number of parallel environments
    )
    env_specific = {
        "CartPole-v1": dict(
            gamma=0.99,           # Standard discount for CartPole
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for CartPole
            minibatch_size=32,    # Smaller batch for faster updates
            episodes_per_epoch=16,# Fewer episodes per epoch for quick feedback
            eval_interval=2,      # Evaluate more frequently for fast convergence
            eval_episodes=10,     # Fewer eval episodes for speed
            reward_threshold=475, # Official CartPole-v1 solved threshold
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for CartPole
            entropy_coef=0.01     # Typical entropy for CartPole
        ),
        "LunarLander-v3": dict(
            gamma=0.99,           # Standard discount for LunarLander
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for LunarLander
            minibatch_size=64,    # Larger batch for more stable updates
            episodes_per_epoch=8, # Fewer episodes per epoch (env is longer)
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=200, # Solved threshold for LunarLander-v3
            policy_lr=1e-4,       # Lower LR for more complex env
            value_lr=5e-4,        # Lower LR for value net
            hidden_dim=128,       # Larger net for more complex env
            entropy_coef=0.02     # Higher entropy for more exploration
        ),
        "Acrobot-v1": dict(
            gamma=0.99,           # Standard discount for Acrobot
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Acrobot
            minibatch_size=32,    # Smaller batch for faster updates
            episodes_per_epoch=16,# Fewer episodes per epoch for quick feedback
            eval_interval=2,      # Evaluate more frequently for fast convergence
            eval_episodes=10,     # Fewer eval episodes for speed
            reward_threshold=-100, # Solved threshold for Acrobot-v1 (average reward > -100)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for Acrobot
            entropy_coef=0.01     # Typical entropy for Acrobot
        ),
        "Pendulum-v1": dict(
            gamma=0.99,           # Standard discount for Pendulum
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Pendulum
            minibatch_size=64,    # Larger batch for continuous action
            episodes_per_epoch=8, # Fewer episodes per epoch (env is longer)
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=-200, # Solved threshold for Pendulum-v1 (average reward > -200)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=128,       # Larger net for continuous control
            entropy_coef=0.0      # No entropy for deterministic continuous control
        ),
        "MountainCar-v0": dict(
            gamma=0.99,             # Discount factor (keep)
            lam=0.97,               # Slightly higher GAE lambda for more bias reduction
            clip_epsilon=0.15,      # Tighter PPO clip for more stable updates
            minibatch_size=16,      # Smaller minibatch for more frequent updates
            episodes_per_epoch=24,  # More episodes per epoch for better sampling
            eval_interval=2,        # Keep frequent evaluation
            eval_episodes=10,       # Keep
            reward_threshold=-110,  # Keep
            policy_lr=1e-4,         # Lower learning rate for more stable policy updates
            value_lr=5e-4,          # Lower value net LR for stability
            hidden_dim=128,         # Larger network for more capacity
            entropy_coef=0.05       # Higher entropy for hard exploration
        ),
    }
    if env_id not in env_specific:
        raise ValueError(f"Unsupported env_id: {env_id}")
    return {**common, **env_specific[env_id]}

ENV_ID = "CartPole-v1"
#ENV_ID = "Acrobot-v1"
#ENV_ID = "LunarLander-v3"
#ENV_ID = "Pendulum-v1"
#ENV_ID = "MountainCar-v0"
CONFIG = setup_config(ENV_ID)
CONFIG

{'env_id': 'CartPole-v1',
 'seed': 42,
 'gamma': 0.99,
 'lam': 0.95,
 'clip_epsilon': 0.2,
 'minibatch_size': 32,
 'episodes_per_epoch': 16,
 'eval_interval': 2,
 'eval_episodes': 10,
 'reward_threshold': 475,
 'policy_lr': 0.0003,
 'value_lr': 0.001,
 'hidden_dim': 64,
 'entropy_coef': 0.01,
 'normalize': False,
 'updates_per_epoch': 10,
 'n_envs': 'auto'}

In [6]:
# Set random seed for reproducibility
set_random_seed(CONFIG['seed'])

# Wrap build env with config parameters
build_env = lambda seed: _build_env(
    CONFIG['env_id'], 
    norm_obs=CONFIG['normalize'], 
    n_envs=CONFIG['n_envs'], seed=seed
)

# Test building env
env = build_env(CONFIG['seed'])
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Action space: Discrete(2)


## 3. Implement PPO Agent
We will define the policy and value networks, and the PPO update step.

In [7]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, act_dim)
        )
    def forward(self, x):
        return self.net(x)

class ValueNet(nn.Module):
    def __init__(self, obs_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, 1)
        )
    def forward(self, x):
        return self.net(x)

In [8]:
from __future__ import annotations

from typing import List, Optional, Tuple

import numpy as np
import torch
from torch.distributions import Categorical

# Local helper ------------------------------------------------------
def _get_device(module: torch.nn.Module) -> torch.device:  # type: ignore
    """Return the device of *module*'s first parameter."""
    return next(module.parameters()).device

def collect_rollouts(
    env,
    policy_model: torch.nn.Module,
    value_model: Optional[torch.nn.Module] = None,
    n_episodes: int = 1,
    deterministic: bool = False,
    collect_frames: bool = False,
    gamma: float = 0.99,
    gae_lambda: float = 0.95,
    return_tensors: Optional[str] = "pt",      # "pt" | "np" | None
    normalize_advantage: bool = True,
    adv_norm_eps: float = 1e-8,
    device=None
):
    if n_episodes < 1:
        raise ValueError("n_episodes must be ≥ 1")
    if return_tensors not in ("pt", "np", None):
        raise ValueError("return_tensors must be 'pt', 'np', or None")

    device = _get_device(policy_model) if device is None else device
    state  = env.reset()
    n_envs = env.num_envs

    # Per-env episode buffers ----------------------------------------
    ep_states      = [[] for _ in range(n_envs)]
    ep_actions     = [[] for _ in range(n_envs)]
    ep_rewards     = [[] for _ in range(n_envs)]
    ep_dones       = [[] for _ in range(n_envs)]
    ep_logps       = [[] for _ in range(n_envs)]
    ep_values      = [[] for _ in range(n_envs)]
    ep_advantages  = [[] for _ in range(n_envs)]
    ep_returns     = [[] for _ in range(n_envs)]
    ep_frames      = [[] for _ in range(n_envs)]            # image sequences

    # Global rollout buffers -----------------------------------------
    (
        states, actions, rewards, dones,
        logps, values, advantages, returns_
    ) = ([] for _ in range(8))
    total_ep_rewards: List[float] = []

    episodes_collected = 0
    # ------------------------------------------------ rollout loop
    while episodes_collected < n_episodes:
        # Policy forward ----------------------------------------
        state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
        with torch.no_grad(): logits  = policy_model(state_t)

        if deterministic:
            action_t = logits.argmax(dim=-1)
            logp_t   = Categorical(logits=logits).log_prob(action_t)  # fast log-p
        else:
            policy_dist = Categorical(logits=logits)
            action_t    = policy_dist.sample()
            logp_t      = policy_dist.log_prob(action_t)

        logp_np  = logp_t.cpu().numpy()
        
        # TODO: just one no_grad()?
        with torch.no_grad(): value_np = value_model(state_t).squeeze(-1).cpu().numpy() if value_model is not None else np.zeros(n_envs, dtype=np.float32)
        
        # Env step ----------------------------------------------
        action_np   = action_t.cpu().numpy()
        next_state, reward, done, _ = env.step(action_np)
        frame_batch = env.get_images() if collect_frames else None

        # Per-env bookkeeping -----------------------------------
        for env_idx in range(n_envs):
            _ep_states      = ep_states[env_idx]
            _ep_actions     = ep_actions[env_idx]
            _ep_rewards     = ep_rewards[env_idx]
            _ep_dones       = ep_dones[env_idx]
            _ep_logps       = ep_logps[env_idx]
            _ep_values      = ep_values[env_idx]
            _ep_advantages  = ep_advantages[env_idx]
            _ep_returns     = ep_returns[env_idx]
            _ep_frames      = ep_frames[env_idx]

            _state  = state[env_idx]
            _action = action_np[env_idx]
            _reward = reward[env_idx]
            _done   = done[env_idx]
            _logp   = logp_np[env_idx]
            _value  = value_np[env_idx]

            # Store ---------------------------------------------
            _ep_states.append(_state)
            _ep_actions.append(int(_action))
            _ep_rewards.append(float(_reward))
            _ep_dones.append(bool(_done))
            _ep_logps.append(float(_logp))
            _ep_values.append(float(_value))
            _ep_advantages.append(0.0)     # placeholder
            _ep_returns   .append(0.0)
            if collect_frames and frame_batch is not None:
                _ep_frames.append(frame_batch[env_idx])

            # Episode still running -----------------------------
            if not _done:
                continue

            # Episode ended -------------------------------------
            total_ep_rewards.append(float(np.sum(_ep_rewards)))

            # -------- GAE advantages ---------------------------
            next_gae   = 0.0
            next_value = 0.0
            for t in reversed(range(len(_ep_rewards))):
                delta              = _ep_rewards[t] + gamma * next_value - _ep_values[t]
                _ep_advantages[t]  = delta + gamma * gae_lambda * next_gae
                next_gae           = _ep_advantages[t]
                next_value         = _ep_values[t]

            # -------- λ-returns via (A + V) --------------------
            for t in range(len(_ep_returns)):
                _ep_returns[t] = _ep_advantages[t] + _ep_values[t]

            # Flush to global buffers ---------------------------
            states     .extend(_ep_states);     _ep_states.clear()
            actions    .extend(_ep_actions);    _ep_actions.clear()
            rewards    .extend(_ep_rewards);    _ep_rewards.clear()
            dones      .extend(_ep_dones);      _ep_dones.clear()
            logps      .extend(_ep_logps);      _ep_logps.clear()
            values     .extend(_ep_values);     _ep_values.clear()
            advantages .extend(_ep_advantages); _ep_advantages.clear()
            returns_   .extend(_ep_returns);    _ep_returns.clear()
            _ep_frames.clear()  # keep structure

            episodes_collected += 1

        state = next_state  # next vector-step

    # -------- optional advantage normalisation (across batch) -------
    if normalize_advantage and advantages:
        adv_np = np.asarray(advantages, dtype=np.float32)
        adv_np = (adv_np - adv_np.mean()) / (adv_np.std() + adv_norm_eps)
        advantages[:] = adv_np.tolist()


    # TODO: extract this code
    def safe_cast(x):
        arr = np.asarray(x)
        if arr.dtype == np.float64:
            arr = arr.astype(np.float32)
        return arr

    if return_tensors == "np":
        convert = lambda x: safe_cast(x)
    elif return_tensors == "pt":
        convert = lambda x: torch.as_tensor(safe_cast(x), device=device)
    else:
        convert = lambda x: safe_cast(x)

    trajectories = tuple(
        convert(buf)
        for buf in (states, actions, rewards, dones, logps, values, advantages, returns_)
    )
    #ep_frames = convert(ep_frames)

    # ----------------------------------------------------------------
    return total_ep_rewards, trajectories, ep_frames

In [ ]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from torch.distributions import Categorical

# TODO: can we infer obs_dim and act_dim from env?
class PPOAgent(pl.LightningModule):
    def __init__(self, obs_dim, act_dim, config):
        super().__init__()
        self.save_hyperparameters()

        # Store and unpack config parameters
        self.config = config
        self.entropy_coef = config['entropy_coef']
        self.clip_epsilon = config['clip_epsilon']
        self.gamma = config['gamma']
        self.lam = config['lam']
        self.episodes_per_epoch = config['episodes_per_epoch']
        self.minibatch_size = config['minibatch_size']
        self.eval_interval = config['eval_interval']
        self.eval_episodes = config['eval_episodes']
        self.reward_threshold = config['reward_threshold']
        self.updates_per_epoch = config['updates_per_epoch']
        self.policy_lr = config['policy_lr']
        self.value_lr = config['value_lr']

        # Initialize policy and value networks
        self.policy_model = PolicyNet(obs_dim, act_dim, hidden_dim=config['hidden_dim'])
        self.value_model = ValueNet(obs_dim, hidden_dim=config['hidden_dim'])

        # Create the environment with the specified seed
        self.env = build_env(config['seed'])
        self.obs_dim = obs_dim
        self.act_dim = act_dim

        # Initialize training rewards list
        self.train_rewards = []

        # Disable automatic optimization to allow manual optimization
        # with multiple optimizers (policy and value networks)
        self.automatic_optimization = False

    def forward(self, x):
        return self.policy_model(x)

    def training_step(self, batch, batch_idx):
        # Manual optimization for multiple optimizers
        opt_policy, opt_value = self.optimizers()
        
        # Collect rollouts with proper device handling
        total_ep_rewards, trajectories, _ = collect_rollouts(
            self.env, 
            self.policy_model, 
            self.value_model, 
            n_episodes=self.episodes_per_epoch,
            deterministic=False, 
            collect_frames=False,
            return_tensors="pt",
            device=self.device
        )
        
        # Unpack trajectories
        states, actions, rewards, dones, logps, values, advantages, returns = trajectories

        num_samples = len(states)
        policy_losses, value_losses = [], []
        
        # PPO update loop
        previous_logps = logps
        for _ in range(self.updates_per_epoch):
            idx = np.random.permutation(num_samples)
            for start in range(0, num_samples, self.minibatch_size):
                end = min(start + self.minibatch_size, num_samples)
                mb_idx = idx[start:end]
                
                # Use policy to select actions for sampled minibatch states
                policy_logits = self.policy_model(states[mb_idx])
                logits_dist = Categorical(logits=policy_logits)
                logps = logits_dist.log_prob(actions[mb_idx])

                # Calculate the ratio between new and old policy probabilities
                # (subtracting log probabilities is the same as dividing probabilities)
                ratio = torch.exp(logps - previous_logps[mb_idx])

                surr1 = ratio * advantages[mb_idx]
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advantages[mb_idx]

                entropy = logits_dist.entropy().mean()
                policy_loss = -torch.min(surr1, surr2).mean() - self.entropy_coef * entropy
                
                # Calculate value loss
                value_pred = self.value_model(states[mb_idx]).squeeze()
                value_loss = ((returns[mb_idx] - value_pred) ** 2).mean()
                
                # Perform policy model optimization step
                opt_policy.zero_grad()
                self.manual_backward(policy_loss)
                opt_policy.step()
                
                # Perform value model optimization step
                opt_value.zero_grad()
                self.manual_backward(value_loss)
                opt_value.step()
                
                policy_losses.append(policy_loss.detach())
                value_losses.append(value_loss.detach())
        
        mean_policy_loss = torch.stack(policy_losses).mean()
        mean_value_loss = torch.stack(value_losses).mean()
        mean_reward = float(np.mean(total_ep_rewards))
        
        self.log('train/mean_reward', mean_reward, prog_bar=True)
        self.log('train/policy_loss', mean_policy_loss)
        self.log('train/value_loss', mean_value_loss)
        self.train_rewards.append(mean_reward)
        
        # Early stopping if solved
        if mean_reward >= self.reward_threshold:
            self.trainer.should_stop = True
            
        return mean_policy_loss + mean_value_loss

    def configure_optimizers(self):
        policy_optim = torch.optim.Adam(self.policy_model.parameters(), lr=self.policy_lr)
        value_optim = torch.optim.Adam(self.value_model.parameters(), lr=self.value_lr)
        return [policy_optim, value_optim]

    def train_dataloader(self):
        # Dummy dataloader: just to trigger training_step
        dummy = torch.zeros(1)
        return DataLoader([dummy], batch_size=1)

## 4. Train PPO Agent
We will train the PPO agent on CartPole-v1.

In [10]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.loggers import WandbLogger

# Create PPO agent and move to device
ppo_agent = PPOAgent(obs_dim, act_dim, CONFIG)

# Set up trainer with proper device configuration
wandb_logger = WandbLogger(project="gymnasium_ppo") # TODO: softcode this

trainer = Trainer(
    logger=wandb_logger,
    #max_epochs=2,
    callbacks=[EarlyStopping(
        monitor='train/mean_reward', 
        mode='max', 
        patience=5, 
        stopping_threshold=CONFIG['reward_threshold']
    )],
    log_every_n_steps=1,
    enable_progress_bar=True,
    accelerator="auto"
)

# Fit the model
trainer.fit(ppo_agent)

# After training, retrieve the trained models
policy_model = ppo_agent.policy_model
value_model = ppo_agent.value_model
train_rewards = ppo_agent.train_rewards


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pytorch_lightning/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.



  | Name         | Type      | Params | Mode 
---------------------------------------------------
0 | policy_model | PolicyNet | 4.6 K  | train
1 | value_model  | ValueNet  | 4.5 K  | train
---------------------------------------------------
9.2 K     Trainable params
0         Non-trainable params
9.2 K     Total params
0.037     Total estimated model params size (MB)
14        Modules in train mode
0         Modules in eval mode
/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 12: 100%|██████████| 1/1 [00:42<00:00,  0.02it/s, v_num=h77a, train/mean_reward=484.0]


In [11]:
from tsilva_notebook_utils.gymnasium import render_episode_frames
import random

ep_rewards, trajectories, ep_frames = collect_rollouts(
    build_env(random.randint(0, 1_000_000)),
    policy_model,
    n_episodes=4,
    deterministic=True,
    collect_frames=True
)
mean_reward = float(np.mean(ep_rewards))
print(f"Mean reward: {mean_reward:.2f}")
render_episode_frames(ep_frames, out_dir="./tmp", grid=(2, 2), text_color=(0, 0, 0))

/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/Users/tsilva/repos/tsilva/aiml-notebooks/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_

Mean reward: 500.00


ValueError: No frames provided.